# Integrated PTRS + PRS — Cross-modal model with per-feature OOF (altPRS config)

Combines the top OneK1K cell-type PTRS (`cd4_naive`) with the top GTEx tissue PTRS
(`Esophagus_Mucosa`) via a **per-feature OOF transform**, then integrates each
feature's calibrated probability with the **altPRS** polygenic scores
(`PRS-CS ϕ=auto, EUR` and `PRS-CSx ϕ=auto, LDREF=META`) using the **Direct** approach
across a panel of 7 classifiers + Rank Addition.

**Pipeline:**

1. Load 2 cross-modal PTRS `Keep_Vector` scores (cd4_naive + Esophagus_Mucosa)
2. Inner-join on GACRS samples; CAMP inner-join across modality-specific external
   sources naturally restricts to CAMP-only (1KG and GTEx don't overlap each other)
3. Global GACRS 75/25 train/test split + z-norm using train stats only
4. **Per-feature OOF transform**: cd4_naive via RF GridSearch, Esophagus_Mucosa via
   Gradient Boosting. 5-fold `cross_val_predict` for OOF on train; full-train fit
   predicts on test / CAMP / CAMP-only. Replaces raw `Keep_Vector` with calibrated
   probability per sample.
5. Load PRS — altPRS config (PRS-CS ϕ=auto + PRS-CSx ϕ=auto, LDREF=META); z-norm
   using train stats only
6. **Direct integration**: feature vector = `[cd4_naive_OOF, Esophagus_OOF, PRS_z]`
   for each PRS variant; run 7 classifiers (LR · Elastic Net CV · RF tuned · GB
   tuned · SVM linear · Stacking LR+RF+GB fixed · Stacking tuned 216-combo grid)
   plus Rank Addition
7. Evaluation: GACRS Test + CAMP-only full + **CAMP-only Balanced (64v64 × 100
   bootstrap)** — primary headline metric

**Outputs** to `${REPO_ROOT}/09_ptrs-unified_model-evaluation/data/predictions/integrated_ptrs_prs_combined/`:

- `all_results.csv` — aggregated metric table (AUC, OR, P-value per method × PRS × eval set)
- `direct_predictions.csv` — per-sample integrated scores
- `per_feature_oof.csv` — per-sample OOF probabilities for each cross-modal feature

> The previous unified PTRS via RF GridSearch step has been **removed** — for the 2-feature
> cross-modal setup, the unified-RF reduction adds nothing over the Direct approach
> (verified empirically: Direct beats Unified across all top-ranked configurations).


## 1. Imports & helpers

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, ElasticNetCV, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.base import clone
from sklearn.metrics import roc_auc_score

from scipy.stats import rankdata, ttest_ind
from sklearn.metrics import average_precision_score

# === Repo root (resolve from notebook location) ===
ROOT = Path.cwd()
print(f"ROOT = {ROOT}")


## 2. Configuration

Cross-modal feature set + altPRS config + the 7 classifiers.

In [ ]:
# ============================================================
# 2.1 Cross-modal feature config
# ============================================================
# Top-1 per modality, picked by CAMP-only Balanced AUC from
# data/predictions/meta_model_{ct,tissue}/consistent_features.csv

# PTRS CSV directories — try both 09 layout (data/) and local layout (data/1/)
def _resolve_ptrs_base(name):
    candidates = [
        ROOT / 'data' / name,           # clean repo: 09/data/<name>/
        ROOT / 'data' / '1' / name,     # local working copy: combine/data/1/<name>/
    ]
    for c in candidates:
        if c.exists():
            return c
    return candidates[0]  # default for error reporting

PTRS_FEATURES = [
    {
        'name': 'cd4_naive',
        'modality': 'ct',  # OneK1K cell type
        'best_per_feature_model': 'RF (GridSearch)',  # from consistent_features.csv
        'gacrs_path': _resolve_ptrs_base('ptrs_results-concat_17CT_gacrs_train_test') / 'cd4_naive_results.csv',
        'camp_path':  _resolve_ptrs_base('ptrs_results-concat_17CT_camp_gtex')        / 'cd4_naive_results.csv',
    },
    {
        'name': 'Esophagus_Mucosa',
        'modality': 'tissue',  # GTEx tissue
        'best_per_feature_model': 'Gradient Boosting',
        'gacrs_path': _resolve_ptrs_base('ptrs_results-concat_39_gacrs_train_test') / 'Esophagus_Mucosa_results.csv',
        'camp_path':  _resolve_ptrs_base('ptrs_results-concat_39_camp_1k1k')        / 'Esophagus_Mucosa_results.csv',
    },
]
for f in PTRS_FEATURES:
    print(f"  {f['name']}: gacrs={f['gacrs_path']}, exists={f['gacrs_path'].exists()}")

# ============================================================
# 2.2 altPRS config
# ============================================================
# PRS-CS:  ϕ=auto, LDREF=EUR (only LDREF option in the file)
# PRS-CSx: ϕ=auto, LDREF=META  — selected by prscs_evaluation.ipynb for cross-population transfer
# Resolve FILES_DIR by checking common staged locations.
def _resolve_files_dir():
    candidates = [
        ROOT / 'data' / 'files',                       # 09 layout: 09/data/files/ (staged here)
        ROOT.parent / 'files',                         # repo-level shared files dir (matches prscs_evaluation convention)
        ROOT.parent / '04_prscs-prscsx-construction' / 'output',  # PRS-CS upstream output
    ]
    for c in candidates:
        if (c / '03_prscs-prscsx-camp-gtex-onek1k-visualization_prscs-gacrs-only-data-melt-admixture.csv').exists():
            return c
    return candidates[0]  # default for error reporting

FILES_DIR = _resolve_files_dir()
print(f"FILES_DIR = {FILES_DIR}")

PRSCS_GACRS_CSV  = FILES_DIR / '03_prscs-prscsx-camp-gtex-onek1k-visualization_prscs-gacrs-only-data-melt-admixture.csv'
PRSCSX_GACRS_CSV = FILES_DIR / '03_prscs-prscsx-camp-gtex-onek1k-visualization_prscsx_gacrs-only-data-melt-admixture.csv'
PRSCS_CAMP_CSV_CANDIDATES  = [
    FILES_DIR / '03_prscs-prscsx-camp-gtex-onek1k-visualization_prscs_camp-gtex-data-melt-admixture.csv',
    FILES_DIR / '03_prscs-prscsx-camp-gtex-onek1k-visualization_prscs_camp-1k1k-data-melt-admixture.csv',
]
PRSCSX_CAMP_CSV_CANDIDATES = [
    FILES_DIR / '03_prscs-prscsx-camp-gtex-onek1k-visualization_prscsx_camp-gtex-data-melt-admixture.csv',
    FILES_DIR / '03_prscs-prscsx-camp-gtex-onek1k-visualization_prscsx_camp-1k1k-data-melt-admixture.csv',
]

PRSCS_PHI    = 'ϕ=auto'
PRSCSX_PHI   = 'ϕ=auto'
PRSCSX_LDREF = 'META'

# ============================================================
# 2.3 Output paths
# ============================================================
OUT_DIR = ROOT / 'data' / 'predictions' / 'integrated_ptrs_prs_combined'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"OUT_DIR = {OUT_DIR}")

# ============================================================
# 2.4 Random seed + bootstrap config
# ============================================================
SEED        = 42
TRAIN_SEED  = 0     # GACRS train/test split seed (preserves prior split)
N_BOOTSTRAP = 100   # CAMP-only Balanced 64v64 bootstrap iterations
BAL_N       = 64


## 3. The 7 integration classifiers + Rank Addition

In [ ]:
# 6 classifiers + Rank Addition (7 total) tried for each Direct (PRS-X) integration
integration_models = {
    'Logistic Regression': LogisticRegression(C=1.0, solver='liblinear', max_iter=1000, random_state=SEED),

    'Elastic Net (CV)': LogisticRegressionCV(
        penalty='elasticnet', l1_ratios=[0.5], solver='saga',
        cv=5, max_iter=2000, random_state=SEED,
    ),

    'Random Forest (tuned)': GridSearchCV(
        RandomForestClassifier(random_state=SEED),
        param_grid={
            'n_estimators':     [50, 100, 200],
            'max_depth':        [2, 3, 5, None],
            'min_samples_leaf': [10, 20, 50],
            'max_features':     [1, 'sqrt', 'log2'],
        },
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
        scoring='roc_auc', n_jobs=-1, refit=True,
    ),

    'Gradient Boosting (tuned)': GridSearchCV(
        GradientBoostingClassifier(random_state=SEED),
        param_grid={
            'n_estimators':  [50, 100],
            'max_depth':     [2, 3],
            'learning_rate': [0.05, 0.1],
        },
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
        scoring='roc_auc', n_jobs=-1, refit=True,
    ),

    'SVM (linear)': CalibratedClassifierCV(
        LinearSVC(C=0.1, max_iter=2000, random_state=SEED), cv=3,
    ),

    'Stacking (LR+RF+GB)': StackingClassifier(
        estimators=[
            ('lr', LogisticRegression(C=0.1, solver='liblinear', max_iter=1000)),
            ('rf', RandomForestClassifier(n_estimators=100, max_depth=3, min_samples_leaf=20, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=50, max_depth=2, learning_rate=0.05, random_state=42)),
        ],
        final_estimator=LogisticRegression(solver='liblinear', max_iter=1000),
        cv=5,
    ),

}
print(f"{len(integration_models)} classifiers configured (+ Rank Addition as 7th non-model baseline)")


## 4. Load 2 cross-modal PTRS features

In [ ]:
# GACRS — inner-join cd4_naive + Esophagus_Mucosa on Sample_ID
feat_names = [f['name'] for f in PTRS_FEATURES]
print(f"Loading cross-modal features: {feat_names}\n")

gacrs_long = []
for f in PTRS_FEATURES:
    d = pd.read_csv(f['gacrs_path'], index_col=0)
    d['Tissue'] = f['name']
    gacrs_long.append(d[['Tissue', 'Keep_Vector', 'asthma']])
gacrs_long = pd.concat(gacrs_long).reset_index()
gacrs_wide = gacrs_long.pivot_table(index='Sample_ID', columns='Tissue', values='Keep_Vector')
gacrs_wide['asthma'] = gacrs_long.drop_duplicates('Sample_ID').set_index('Sample_ID')['asthma']
before = len(gacrs_wide)
gacrs_wide = gacrs_wide.dropna(subset=feat_names)
print(f"GACRS samples with both features: {len(gacrs_wide)} (dropped {before - len(gacrs_wide)})")
print(f"  Cases: {int(gacrs_wide['asthma'].sum())}, Controls: {int((gacrs_wide['asthma']==0).sum())}")

# CAMP — inner-join across modality-specific external sources →
# naturally restricts to CAMP-only samples (1KG and GTEx don't overlap)
camp_long = []
for f in PTRS_FEATURES:
    d = pd.read_csv(f['camp_path'], index_col=0)
    d['Tissue'] = f['name']
    camp_long.append(d[['Tissue', 'Keep_Vector', 'asthma']])
camp_long = pd.concat(camp_long).reset_index()
camp_wide = camp_long.pivot_table(index='Sample_ID', columns='Tissue', values='Keep_Vector')
camp_wide['asthma'] = camp_long.drop_duplicates('Sample_ID').set_index('Sample_ID')['asthma']
before = len(camp_wide)
camp_wide = camp_wide.dropna(subset=feat_names + ['asthma'])
print(f"\nCAMP samples with both features (effectively CAMP-only): {len(camp_wide)} (dropped {before - len(camp_wide)})")
print(f"  Cases: {int(camp_wide['asthma'].sum())}, Controls: {int((camp_wide['asthma']==0).sum())}")

combined_features = sorted(feat_names)


## 5. GACRS 75/25 train/test split + z-norm features (train stats only)

In [ ]:
all_samples = gacrs_wide.index
all_labels  = gacrs_wide['asthma']
train_ids, test_ids = train_test_split(
    all_samples, test_size=0.25, stratify=all_labels, random_state=TRAIN_SEED,
)
train_id = train_ids.values
test_id  = test_ids.values
print(f"Train: {len(train_id)} ({int(gacrs_wide.loc[train_id, 'asthma'].sum())} cases)")
print(f"Test:  {len(test_id)} ({int(gacrs_wide.loc[test_id, 'asthma'].sum())} cases)")

# Z-norm feature columns using train stats only
for col in combined_features:
    m = gacrs_wide.loc[train_id, col].mean()
    s = gacrs_wide.loc[train_id, col].std()
    gacrs_wide[col] = (gacrs_wide[col] - m) / s
    camp_wide[col]  = (camp_wide[col]  - m) / s

X_train     = gacrs_wide.loc[train_id, combined_features]
X_test      = gacrs_wide.loc[test_id,  combined_features]
y_train     = gacrs_wide.loc[train_id, 'asthma']
y_test      = gacrs_wide.loc[test_id,  'asthma']
X_camp      = camp_wide[combined_features]
y_camp      = camp_wide['asthma']
print(f"\nFeatures z-normed using GACRS train stats only.")


## 6. Per-feature OOF transform

For each cross-modal feature, fit its declared best per-feature model on the
feature's 1-D Keep_Vector column. Use 5-fold `cross_val_predict` for the
train OOF probability (no train-label leak); full-train fit predicts on test
and CAMP. The OOF/predicted probability **replaces** the raw Keep_Vector in
the feature matrices.

In [ ]:
import re

def make_per_feature_estimator(model_name):
    if 'RF' in model_name:
        return GridSearchCV(
            RandomForestClassifier(random_state=SEED),
            param_grid={
                'n_estimators':     [50, 100, 200],
                'max_depth':        [2, 3, 5, None],
                'min_samples_leaf': [10, 20, 50],
                'max_features':     [1],  # 1-D input
            },
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
            scoring='roc_auc', n_jobs=-1, refit=True,
        )
    if 'Gradient Boosting' in model_name:
        return GradientBoostingClassifier(random_state=SEED)
    if 'Ridge' in model_name or 'Lasso' in model_name:
        m = re.search(r'C=([0-9.]+)', model_name)
        C = float(m.group(1)) if m else 1.0
        penalty = 'l1' if 'Lasso' in model_name else 'l2'
        return LogisticRegression(penalty=penalty, C=C, solver='liblinear', max_iter=1000, random_state=SEED)
    raise ValueError(f"unsupported per-feature model: {model_name}")

feat_to_model = {f['name']: f['best_per_feature_model'] for f in PTRS_FEATURES}
print('Per-feature transforms:')
for f in combined_features:
    print(f"  {f:30s} -> {feat_to_model[f]}")

# Hold OOF/predicted probabilities in a long table for save
oof_rows = []
skf_pf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for feat in combined_features:
    est = make_per_feature_estimator(feat_to_model[feat])
    Xtr1d = X_train[[feat]].values
    Xte1d = X_test[[feat]].values
    Xcm1d = X_camp[[feat]].values
    ytr = y_train.values

    if isinstance(est, GridSearchCV):
        est.fit(Xtr1d, ytr)
        best = est.best_estimator_
        print(f"\n  [{feat}] best RF params: {est.best_params_}")
    else:
        best = est

    oof_tr  = cross_val_predict(best, Xtr1d, ytr, cv=skf_pf, method='predict_proba')[:, 1]
    best.fit(Xtr1d, ytr)
    pred_te = best.predict_proba(Xte1d)[:, 1]
    pred_cm = best.predict_proba(Xcm1d)[:, 1]

    # Sanity AUCs (single-feature performance through its declared best model)
    print(f"  [{feat}]  train OOF AUC = {roc_auc_score(ytr, oof_tr):.4f}, "
          f"test AUC = {roc_auc_score(y_test, pred_te):.4f}, "
          f"CAMP AUC = {roc_auc_score(y_camp, pred_cm):.4f}")

    # Replace in feature matrices (probabilities instead of z-normed Keep_Vector)
    gacrs_wide.loc[train_id, feat] = oof_tr
    gacrs_wide.loc[test_id,  feat] = pred_te
    camp_wide[feat]                = pred_cm

    # Stash for save
    for sid, p, y in zip(train_id, oof_tr, y_train.values):
        oof_rows.append({'sample_id': sid, 'feature': feat, 'cohort': 'GACRS_train_OOF',
                         'score': p, 'y_true': int(y)})
    for sid, p, y in zip(test_id, pred_te, y_test.values):
        oof_rows.append({'sample_id': sid, 'feature': feat, 'cohort': 'GACRS_test',
                         'score': p, 'y_true': int(y)})
    for sid, p, y in zip(camp_wide.index, pred_cm, y_camp.values):
        oof_rows.append({'sample_id': sid, 'feature': feat, 'cohort': 'CAMP_only',
                         'score': p, 'y_true': int(y)})

# Refresh feature matrices after replacement
X_train = gacrs_wide.loc[train_id, combined_features]
X_test  = gacrs_wide.loc[test_id,  combined_features]
X_camp  = camp_wide[combined_features]

per_feature_oof_df = pd.DataFrame(oof_rows)
per_feature_oof_df.to_csv(OUT_DIR / 'per_feature_oof.csv', index=False)
print(f"\nSaved per-feature OOF probabilities → {OUT_DIR / 'per_feature_oof.csv'} ({len(per_feature_oof_df)} rows)")
print(f"X_train range now: [{X_train.values.min():.3f}, {X_train.values.max():.3f}]")


## 7. Load PRS — altPRS config

- **PRS-CS** at ϕ=auto (LDREF is EUR — the only LDREF option for single-population PRS-CS)
- **PRS-CSx** at ϕ=auto, LDREF=META

Both z-normalized using GACRS train stats only.

In [ ]:
# --- PRS-CS ---
prscs_gacrs_full = pd.read_csv(PRSCS_GACRS_CSV)
phi_col_cs = [c for c in prscs_gacrs_full.columns if 'ϕ' in c][0]
prscs_camp_path = next(p for p in PRSCS_CAMP_CSV_CANDIDATES if p.exists())
print(f"PRS-CS CAMP file: {prscs_camp_path.name}")
prscs_camp_full = pd.read_csv(prscs_camp_path)

prscs_df = prscs_gacrs_full[prscs_gacrs_full[phi_col_cs] == PRSCS_PHI][['IID', 'PRS']].rename(
    columns={'IID': 'Sample_ID', 'PRS': 'PRS_CS'}).set_index('Sample_ID')
prscs_camp_df = prscs_camp_full[prscs_camp_full[phi_col_cs] == PRSCS_PHI][['IID', 'PRS']].rename(
    columns={'IID': 'Sample_ID', 'PRS': 'PRS_CS'}).set_index('Sample_ID')

# --- PRS-CSx ---
prscsx_gacrs_full = pd.read_csv(PRSCSX_GACRS_CSV)
phi_col_csx = [c for c in prscsx_gacrs_full.columns if 'ϕ' in c][0]
prscsx_camp_path = next(p for p in PRSCSX_CAMP_CSV_CANDIDATES if p.exists())
print(f"PRS-CSx CAMP file: {prscsx_camp_path.name}")
prscsx_camp_full = pd.read_csv(prscsx_camp_path)

prscsx_df = prscsx_gacrs_full[
    (prscsx_gacrs_full[phi_col_csx] == PRSCSX_PHI) & (prscsx_gacrs_full['LDREF'] == PRSCSX_LDREF)
][['IID', 'PRS']].rename(columns={'IID': 'Sample_ID', 'PRS': 'PRS_CSx'}).set_index('Sample_ID')
prscsx_camp_df = prscsx_camp_full[
    (prscsx_camp_full[phi_col_csx] == PRSCSX_PHI) & (prscsx_camp_full['LDREF'] == PRSCSX_LDREF)
][['IID', 'PRS']].rename(columns={'IID': 'Sample_ID', 'PRS': 'PRS_CSx'}).set_index('Sample_ID')

print(f"\nPRS-CS  GACRS: {len(prscs_df)}, CAMP: {len(prscs_camp_df)}")
print(f"PRS-CSx GACRS: {len(prscsx_df)}, CAMP: {len(prscsx_camp_df)}")

# Merge + z-norm using train stats
prs_df  = prscs_df.merge(prscsx_df, left_index=True, right_index=True, how='inner')
prs_camp = prscs_camp_df.merge(prscsx_camp_df, left_index=True, right_index=True, how='inner')
print(f"\nPRS in GACRS: {len(prs_df)}, PRS in CAMP: {len(prs_camp)}")

valid_train_prs = [s for s in train_id if s in prs_df.index]
for col in ['PRS_CS', 'PRS_CSx']:
    m = prs_df.loc[valid_train_prs, col].mean()
    s = prs_df.loc[valid_train_prs, col].std()
    prs_df[col + '_z']  = (prs_df[col]  - m) / s
    prs_camp[col + '_z'] = (prs_camp[col] - m) / s
print(f"\nPRS z-normed using GACRS train stats only.")


## 8. Build integrated DataFrames

In [ ]:
# Merge the OOF-transformed feature matrix with the z-normed PRS columns
df_int_gacrs = gacrs_wide[['asthma'] + combined_features].merge(
    prs_df[['PRS_CS_z', 'PRS_CSx_z']], left_index=True, right_index=True, how='inner')
df_int_camp  = camp_wide[['asthma'] + combined_features].merge(
    prs_camp[['PRS_CS_z', 'PRS_CSx_z']], left_index=True, right_index=True, how='inner')

valid_train_int = [s for s in train_id if s in df_int_gacrs.index]
valid_test_int  = [s for s in test_id  if s in df_int_gacrs.index]
print(f"Integrated GACRS train: {len(valid_train_int)}")
print(f"Integrated GACRS test:  {len(valid_test_int)}")
print(f"Integrated CAMP:        {len(df_int_camp)}")

y_int_train = df_int_gacrs.loc[valid_train_int, 'asthma']
y_int_test  = df_int_gacrs.loc[valid_test_int,  'asthma']
y_int_camp  = df_int_camp['asthma']


## 9. Evaluation helpers

In [ ]:
def odds_ratio_quantile(y_true, y_pred, q=0.25):
    """Quartile-based OR: top vs bottom quartile of predicted score."""
    thresh_high = np.quantile(y_pred, 1 - q)
    thresh_low  = np.quantile(y_pred, q)
    top    = y_true[y_pred >= thresh_high]
    bottom = y_true[y_pred <= thresh_low]
    a = top.sum();    b = len(top) - a
    c = bottom.sum(); d = len(bottom) - c
    if b == 0 or c == 0:
        return np.inf
    return (a * d) / (b * c)


def evaluate(name, eval_set, y_true, scores, results, n_repeats=N_BOOTSTRAP, balanced=False):
    """Append a metric row to `results`. CAMP-only Balanced bootstraps 100×64-vs-64
    and reports mean±SD of AUC and OR per iteration. Matches the convention used by
    the architecture-search experiments (notebooks under data/predictions/stacking_experiment_*)."""
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    if balanced:
        case_idx    = np.where(y_true == 1)[0]
        control_idx = np.where(y_true == 0)[0]
        # Match the original convention: downsample cases to len(controls) — i.e.
        # 64 case vs 64 control where CAMP has 64 controls.
        n_target = len(control_idx)
        aucs, ors, diffs = [], [], []
        for seed in range(n_repeats):
            rng = np.random.RandomState(seed)
            cs  = rng.choice(case_idx, size=n_target, replace=False)
            idx = np.concatenate([cs, control_idx])
            y_b = y_true[idx]; p_b = scores[idx]
            aucs.append(roc_auc_score(y_b, p_b))
            ors.append(odds_ratio_quantile(y_b, p_b))
            diffs.append(p_b[y_b == 1].mean() - p_b[y_b == 0].mean())
        results.append({'Method': name, 'Eval_Set': eval_set,
                        'AUC': float(np.mean(aucs)), 'AUC_std': float(np.std(aucs)),
                        'OR': float(np.mean(ors)), 'OR_std': float(np.std(ors)),
                        'Mean_Diff': float(np.mean(diffs)),
                        'AUPRC': np.nan, 'P_Value': np.nan,
                        'N_cases': n_target, 'N_controls': n_target})
    else:
        auc   = roc_auc_score(y_true, scores)
        auprc = average_precision_score(y_true, scores)
        OR    = odds_ratio_quantile(y_true, scores)
        g0 = scores[y_true == 0]; g1 = scores[y_true == 1]
        if len(g0) > 0 and len(g1) > 0:
            _, p = ttest_ind(g0, g1, equal_var=False)  # Welch two-sample t-test
            mean_diff = g1.mean() - g0.mean()
        else:
            p = np.nan; mean_diff = np.nan
        results.append({'Method': name, 'Eval_Set': eval_set,
                        'AUC': auc, 'AUC_std': np.nan,
                        'OR': OR, 'OR_std': np.nan,
                        'Mean_Diff': mean_diff, 'AUPRC': auprc,
                        'P_Value': float(p) if not np.isnan(p) else np.nan,
                        'N_cases': int(y_true.sum()), 'N_controls': int((y_true == 0).sum())})


def append_pred(rows, sample_ids, scores, y_true, method, prs_type, eval_set):
    for sid, s, y in zip(sample_ids, scores, y_true):
        rows.append({'sample_id': sid, 'score': float(s), 'y_true': int(y),
                     'method': method, 'prs_type': prs_type, 'eval_set': eval_set})


## 10. Direct integration with all 7 classifiers + Rank Addition

For each PRS variant (PRS-CS, PRS-CSx), build feature vector
`[cd4_naive_OOF, Esophagus_OOF, PRS_X_z]` and run every classifier.

In [ ]:
results_direct = []
direct_predictions = []

for prs_label, prs_col in [('PRS-CS', 'PRS_CS_z'), ('PRS-CSx', 'PRS_CSx_z')]:
    features = combined_features + [prs_col]
    Xtr   = df_int_gacrs.loc[valid_train_int, features]
    Xte   = df_int_gacrs.loc[valid_test_int,  features]
    Xcamp = df_int_camp[features]
    print(f"\n=== Direct ({prs_label}) — {len(features)} features ===")

    for method_name, model_template in integration_models.items():
        model = clone(model_template)
        model.fit(Xtr, y_int_train)
        test_p = model.predict_proba(Xte)[:, 1]
        camp_p = model.predict_proba(Xcamp)[:, 1]
        full_name = f"Direct ({prs_label}) + {method_name}"

        evaluate(full_name, 'GACRS Test',         y_int_test.values, test_p, results_direct)
        evaluate(full_name, 'CAMP-only (full)',   y_int_camp.values, camp_p, results_direct)
        evaluate(full_name, 'CAMP-only Balanced', y_int_camp.values, camp_p, results_direct, balanced=True)

        append_pred(direct_predictions, Xte.index.tolist(),   test_p, y_int_test.values,
                    method=method_name, prs_type=prs_label, eval_set='GACRS Test')
        append_pred(direct_predictions, Xcamp.index.tolist(), camp_p, y_int_camp.values,
                    method=method_name, prs_type=prs_label, eval_set='CAMP-only')
        print(f"  ✓ {method_name}")

    # Rank Addition — equal-weight rank-sum of (cd4_naive, Esophagus, PRS)
    def rank_norm(s): return rankdata(s) / len(s)
    def rank_sum(X): return sum(rank_norm(X[c].values) for c in features)
    test_rs = rank_sum(Xte); camp_rs = rank_sum(Xcamp)
    name_ra = f"Direct ({prs_label}) + Rank Addition"
    evaluate(name_ra, 'GACRS Test',         y_int_test.values, test_rs, results_direct)
    evaluate(name_ra, 'CAMP-only (full)',   y_int_camp.values, camp_rs, results_direct)
    evaluate(name_ra, 'CAMP-only Balanced', y_int_camp.values, camp_rs, results_direct, balanced=True)
    append_pred(direct_predictions, Xte.index.tolist(),   test_rs, y_int_test.values,
                method='Rank Addition', prs_type=prs_label, eval_set='GACRS Test')
    append_pred(direct_predictions, Xcamp.index.tolist(), camp_rs, y_int_camp.values,
                method='Rank Addition', prs_type=prs_label, eval_set='CAMP-only')
    print(f"  ✓ Rank Addition")

results_direct_df = pd.DataFrame(results_direct)
print(f"\nDirect integration: {len(results_direct_df)} result rows")


## 11. PRS-only baselines (for reference)

In [ ]:
# Logistic Regression on a single PRS column — no PTRS, no transform
results_prs_only = []
for prs_col, prs_label in [('PRS_CS_z', 'PRS-CS'), ('PRS_CSx_z', 'PRS-CSx')]:
    cols  = [prs_col]
    Xtr   = df_int_gacrs.loc[valid_train_int, cols]
    Xte   = df_int_gacrs.loc[valid_test_int,  cols]
    Xcamp = df_int_camp[cols]

    model = LogisticRegression(solver='liblinear', max_iter=1000, random_state=SEED)
    model.fit(Xtr, y_int_train)
    test_p = model.predict_proba(Xte)[:, 1]
    camp_p = model.predict_proba(Xcamp)[:, 1]

    name = f"{prs_label} only"
    evaluate(name, 'GACRS Test',         y_int_test.values, test_p, results_prs_only)
    evaluate(name, 'CAMP-only (full)',   y_int_camp.values, camp_p, results_prs_only)
    evaluate(name, 'CAMP-only Balanced', y_int_camp.values, camp_p, results_prs_only, balanced=True)

results_prs_only_df = pd.DataFrame(results_prs_only)
print(results_prs_only_df.to_string(index=False))


## 12. Per-feature alone baselines (for reference)

Each cross-modal feature's calibrated probability evaluated by itself (no PRS,
no other feature).

In [ ]:
results_feat_only = []
for feat in combined_features:
    # Already have OOF/predicted probability in gacrs_wide and camp_wide
    test_p = gacrs_wide.loc[test_id, feat].values
    camp_p = camp_wide[feat].values
    name = f"{feat} alone (per-feature OOF)"
    evaluate(name, 'GACRS Test',         y_test.values, test_p, results_feat_only)
    evaluate(name, 'CAMP-only (full)',   y_camp.values, camp_p, results_feat_only)
    evaluate(name, 'CAMP-only Balanced', y_camp.values, camp_p, results_feat_only, balanced=True)

results_feat_only_df = pd.DataFrame(results_feat_only)
print(results_feat_only_df.to_string(index=False))


## 13. Summary + save outputs

In [ ]:
all_results = pd.concat([
    results_direct_df.assign(Approach='Direct'),
    results_prs_only_df.assign(Approach='PRS only'),
    results_feat_only_df.assign(Approach='Per-feature only'),
], ignore_index=True)

# Per-eval-set ranking
print("=== CAMP-only Balanced — sorted by AUC desc ===")
sub = all_results[all_results['Eval_Set']=='CAMP-only Balanced'].sort_values('AUC', ascending=False)
print(sub[['Approach', 'Method', 'AUC', 'AUC_std', 'OR']].to_string(index=False))

# Best method per eval set
print("\n=== Best method per evaluation set ===")
for es in all_results['Eval_Set'].unique():
    sub = all_results[all_results['Eval_Set']==es]
    best = sub.loc[sub['AUC'].idxmax()]
    extra = f" ± {best['AUC_std']:.4f}" if not pd.isna(best['AUC_std']) else ''
    print(f"  {es}: {best['Approach']} | {best['Method']} — AUC={best['AUC']:.4f}{extra}")

# Save
all_results.to_csv(OUT_DIR / 'all_results.csv', index=False)
print(f"\nSaved {len(all_results)} result rows → {OUT_DIR / 'all_results.csv'}")

direct_pred_df = pd.DataFrame(direct_predictions)
direct_pred_df.to_csv(OUT_DIR / 'direct_predictions.csv', index=False)
print(f"Saved {len(direct_pred_df)} per-sample predictions → {OUT_DIR / 'direct_predictions.csv'}")


## 14. Headline rank-1 winner

The expected winner is `Direct (PRS-CSx) + Random Forest (tuned)` at ~0.632 ± 0.040
on CAMP-only Balanced — the rank-1 model across the full architecture-search.

In [ ]:
rank1 = all_results[all_results['Eval_Set']=='CAMP-only Balanced'].sort_values('AUC', ascending=False).iloc[0]
print("=== RANK-1 model on CAMP-only Balanced ===")
print(f"  Approach : {rank1['Approach']}")
print(f"  Method   : {rank1['Method']}")
print(f"  CAMP-bal AUC = {rank1['AUC']:.4f} ± {rank1['AUC_std']:.4f}")
print(f"  OR           = {rank1['OR']:.2f}")
print(f"  N           = {rank1['N_cases']} cases × {rank1['N_controls']} controls (per bootstrap)")
